# Thai Sentiment — FastText Training with MLflow

This notebook trains a FastText supervised classifier on the preprocessed Wisesight sentiment splits, logging parameters, metrics, and artifacts to MLflow.

## 1. Setup & Environment
Reinstall numpy at a compatible version, install remaining dependencies, mount Google Drive, and set the working directory.

In [1]:
!pip uninstall -y numpy

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


In [2]:
!pip install fasttext-wheel mlflow pythainlp boto3 python-dotenv numpy==1.26.4 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Imports

In [5]:
import os
import logging
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fasttext
import mlflow
from dotenv import load_dotenv
from pythainlp.tokenize import word_tokenize
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

## 3. Load Environment Variables
Loads `.env` (e.g. `MLFLOW_TRACKING_URI`).

In [6]:
load_dotenv('.env')

True

## 4. Logging Configuration

In [7]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 5. Configuration
Label mapping and file paths for the FastText-formatted data and model output.

In [8]:
LABEL_NAMES = ["neg", "neu", "pos", "q"]
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
FASTTEXT_TRAIN_FILE = "/tmp/ft_train.txt"
FASTTEXT_VAL_FILE   = "/tmp/ft_val.txt"
FASTTEXT_TEST_FILE  = "/tmp/ft_test.txt"
MODEL_OUT_PATH      = "/tmp/fasttext_thai_sentiment"

## 6. Helper Functions

### 6.1 Thai Tokenizer
Wraps PyThaiNLP's `newmm` tokenizer.

In [9]:
# ── PyThaiNLP tokenizer ──
def thai_tokenize(text: str) -> str:
    tokens = word_tokenize(str(text), engine="newmm", keep_whitespace=False)
    return " ".join(tokens)

### 6.2 Write FastText-Formatted File
Converts a DataFrame into FastText's `__label__<class> <tokens>` text format.

In [10]:
def write_fasttext_file(df: pd.DataFrame, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            label     = ID2LABEL[int(row["label"])]
            tokenized = thai_tokenize(row["text_clean"])
            f.write(f"__label__{label} {tokenized}\n")
    logger.info(f"Wrote {len(df):,} samples → {path}")

### 6.3 Predict Helper
Runs a FastText model over a list of texts and maps predictions back to label IDs.

In [11]:
def predict(model, texts: list[str]) -> list[int]:
    preds = []

    for text in texts:
        tokenized = thai_tokenize(text)

        # fastText prediction
        labels, probs = model.predict(tokenized)

        # get first label
        label_str = labels[0]

        # decode bytes if needed
        if isinstance(label_str, bytes):
            label_str = label_str.decode("utf-8")

        # remove prefix
        label_name = label_str.replace("__label__", "")

        # convert to id
        preds.append(LABEL2ID[label_name])

    return preds

### 6.4 Confusion Matrix Plotter

In [12]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> plt.Figure:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.tight_layout()
    return fig

## 7. Load Data
Read the cleaned train/val/test CSVs produced by the preprocessing notebook.

In [13]:
train = pd.read_csv("data/processed/train.csv")
val   = pd.read_csv("data/processed/val.csv")
test  = pd.read_csv("data/processed/test.csv")

### 7.1 Split Sizes

In [14]:
logger.info(f"Train : {len(train):,}")
logger.info(f"Val   : {len(val):,}")
logger.info(f"Test  : {len(test):,}")

## 8. Tokenize & Write FastText Files
Convert each split into FastText's expected input format on disk.

In [15]:
# ── Tokenize + write FastText files ──
logger.info("Writing FastText format files...")
write_fasttext_file(train, FASTTEXT_TRAIN_FILE)
write_fasttext_file(val,   FASTTEXT_VAL_FILE)
write_fasttext_file(test,  FASTTEXT_TEST_FILE)
logger.info("Done ✅")

## 9. Training Function
Trains a FastText supervised model, evaluates on val/test, and logs params, metrics, confusion matrix, classification report, and the model artifact to MLflow.

In [16]:
def train_fasttext(
    lr:        float = 0.5,
    epoch:     int   = 25,
    wordNgrams:int   = 2,
    dim:       int   = 100,
    minCount:  int   = 2,
    loss:      str   = "softmax",
):
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
    mlflow.set_experiment("thai-sentiment / FastText")

    with mlflow.start_run(run_name="fasttext"):

        # ── Log parameters ──
        mlflow.log_params({
            "model_type":    "fasttext",
            "tokenizer":     "pythainlp_newmm",
            "lr":            lr,
            "epoch":         epoch,
            "wordNgrams":    wordNgrams,
            "dim":           dim,
            "minCount":      minCount,
            "loss":          loss,
            "train_samples": len(train),
            "val_samples":   len(val),
            "test_samples":  len(test),
        })
        mlflow.set_tags({
            "method":   "FastText",
            "language": "thai",
            "dataset":  "wisesight_sentiment",
        })

        # ── Train ──
        logger.info("Training FastText...")
        model = fasttext.train_supervised(
            input=FASTTEXT_TRAIN_FILE,
            lr=lr,
            epoch=epoch,
            wordNgrams=wordNgrams,
            dim=dim,
            minCount=minCount,
            loss=loss,
            thread=os.cpu_count(),
            verbose=2,
        )

        # ── Validate ──
        val_preds  = predict(model, val["text_clean"].tolist())
        val_labels = val["label"].tolist()
        val_acc    = accuracy_score(val_labels, val_preds)
        val_f1     = f1_score(val_labels, val_preds, average="weighted")
        mlflow.log_metrics({
            "val_accuracy":    val_acc,
            "val_f1_weighted": val_f1,
        })
        logger.info(f"Val → accuracy: {val_acc:.4f}  F1: {val_f1:.4f}")

        # ── Test ──
        test_preds  = predict(model, test["text_clean"].tolist())
        test_labels = test["label"].tolist()
        test_acc    = accuracy_score(test_labels, test_preds)
        test_f1_w   = f1_score(test_labels, test_preds, average="weighted")
        test_f1_m   = f1_score(test_labels, test_preds, average="macro")
        mlflow.log_metrics({
            "test_accuracy":    test_acc,
            "test_f1_weighted": test_f1_w,
            "test_f1_macro":    test_f1_m,
        })
        logger.info(f"Test → accuracy: {test_acc:.4f}  F1: {test_f1_w:.4f}")

        # ── Confusion matrix ──
        fig = plot_confusion_matrix(test_labels, test_preds, "FastText")
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        # ── Classification report ──
        report = classification_report(
            test_labels, test_preds, target_names=LABEL_NAMES
        )
        mlflow.log_text(report, "classification_report.txt")
        print(report)

        # ── Save + log model file ──
        model_path = MODEL_OUT_PATH + ".bin"
        model.save_model(model_path)
        mlflow.log_artifact(model_path, artifact_path="model")
        logger.info(f"Model saved → {model_path}")

        run_id = mlflow.active_run().info.run_id
        logger.info(f"MLflow run ID: {run_id}")

    return model, {
        "test_accuracy":    test_acc,
        "test_f1_weighted": test_f1_w,
        "test_f1_macro":    test_f1_m,
    }

## 10. Run Training

In [17]:
# ── Run ──
model, results = train_fasttext(
    lr=0.5,
    epoch=25,
    wordNgrams=2,
    dim=100,
    loss="softmax",
)

print("\n=== Final Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

              precision    recall  f1-score   support

         neg       0.52      0.42      0.46       478
         neu       0.72      0.80      0.76      1453
         pos       0.76      0.70      0.73       683
           q       0.39      0.28      0.33        57

    accuracy                           0.70      2671
   macro avg       0.60      0.55      0.57      2671
weighted avg       0.69      0.70      0.69      2671

🏃 View run fasttext at: http://43.210.51.34:5000/#/experiments/5/runs/8fcd6309b23445eb828822eaf3b41500
🧪 View experiment at: http://43.210.51.34:5000/#/experiments/5

=== Final Results ===
test_accuracy: 0.6956
test_f1_weighted: 0.6888
test_f1_macro: 0.5692
